In [1]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [2]:
layer = 1
features = [1, 6243, 19991, 8434, 8231]

In [3]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [4]:
def load_sae(
    layer: int
):
    local_dir = "../jbloom_dictionaries"
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")

    return sae

In [5]:
from llama_config import EnvConfig, ExplainerConfig, CondenserConfig, DetectionScorerConfig, GenerationScorerConfig

env_config = EnvConfig()
explainer_cfg = ExplainerConfig()
condenser_cfg = CondenserConfig()
d_scorer_cfg = DetectionScorerConfig()
gen_scorer_cfg = GenerationScorerConfig()

env = Environment(
    model=model, 
    sae=load_sae(layer),
    cfg=env_config,
    provider="replicate"
)

env.load_cache(layer)

100%|██████████| 15/15 [00:46<00:00,  3.10s/it]

Activation Cache Size: torch.Size([2250, 128, 24576])


In [6]:
top = {}

threshold = 0.2

for feature in features:
    top_examples, max_act = env.get_top_examples(feature)[:5]

    decoded = []
    for example in top_examples:
        example_toks, example_acts = example

        delimited_string = ''
        for pos in range(example_toks.size(0)):
            if example_acts[pos] > (threshold * max_act):
                delimited_string += "[HIGHLIGHT]" + model.tokenizer.decode(example_toks[pos]) + "[/HIGHLIGHT]"
            else:
                delimited_string += model.tokenizer.decode(example_toks[pos])
        
        decoded.append(delimited_string)

    top[feature] = decoded

import json

with open(f"./results/{layer}_top_examples.json", "w") as f:
    json.dump(top, f, indent=4)


: 